In [ ]:
import sys
sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction')
# sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers')

%run "../helpers/data_manipulator.py"
%run "../helpers/yfinance_data_fetcher.py"

from custom_objects import register_custom_objects
# Register globally
register_custom_objects()

### LOAD VECTORS FOR TODAY

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
from keras.models import load_model

combos = [
    ["AAPL", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AAPL", "experimental_v3", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AMD", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 1],
    ["AMZN", "experimental_v3", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["NVDA", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["SPOT", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["SPOT", "experimental_v3", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["SPY", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 3],
    ["SPY", "experimental_v3", "Next-Day-High-To-Next-Day-Open-Ratio", 2],
    ["TSLA", "experimental_v2", "Next-Day-High-To-Next-Day-Open-Ratio", 4],
    ["TSLA", "experimental_v3", "Next-Day-High-To-Next-Day-Open-Ratio", 4],
                   ]

write_header = True
for combo in combos:
    ss = combo[0]
    mm = combo[1]
    tt = combo[2]
    print(f"SS: {ss}, MM: {mm} TT: {tt}")
    score = combo[3]
    # Prepare model location and name
    model_location = "../models-feb-update-experimental/" + ss + "/cnn/" + mm + "/"
    model_name = tt + ".keras"
    threshold = pd.read_csv(model_location + tt + "no-quantile_threshold.csv").iloc[0]["Threshold"]
    lvls = pd.read_csv(model_location + tt + "no-quantile_all_thresholds.csv")
    threshold_levels = lvls.iloc[(lvls['threshold'] - threshold).abs().idxmin()]
    print(f"\n\n\n\n threshold_levels: {threshold_levels}")
    target_val = pd.read_csv(model_location + tt + "_binary_target_val.csv").iloc[0]["Target-Val"]

    save_location = "../predictions-feb-update-experimental/"
    from pathlib import Path
    directory = Path(save_location)
    directory.mkdir(parents=True, exist_ok=True)

    print(f"stock: {ss}, target: {tt}, threshold: {threshold}, target_val: {target_val}")
    
    # Load the best performing model
    best_model = load_model(model_location + model_name)
    
    #data_frame = construct_values_for_model(ticker_symbols = stocks_to_check, sequence_size=20, use_for_last_day_prediction=True, verbose=False, refresh=True)
    data_frame, real_target = construct_values_for_model(ticker_symbols = [ss], 
                                                        underlying_target=tt,
                                                        sequence_size=dm_sequence_size(mm), 
                                                        use_for_last_day_prediction=False, 
                                                        verbose=False, 
                                                        data_interval="2y",
                                                        refresh=True,
                                                        is_binary_prediction=True,
                                                        incl_earnings=True
                                                        )

    print(f"all data : {data_frame}")
    data_frame = get_specific_date_data(data_frame, "2025-02-25")
    # data_frame = get_specific_date_internal_data(data_frame, "2025-01-01", "2025-01-30")

    i = True
    if i:
        print(data_frame)
        print(data_frame["LstmData"])
        if pd.isna(data_frame.iloc[-1]['Next-Day-Open']):
                print(f"ABORTING.... NEXT DAY'S DATA IS NOT AVAILABLE FOR THE LAST DAY IN YOUR SELECTION...")
                break
        i = False
    
    # print(data_frame)
    X_test = np.array(data_frame["LstmData"].to_list())
    X_test_ticker = np.array(data_frame["Ticker"].to_list())
    X_test_sector = np.array(data_frame["Sector"].to_list())
    X_test_yesterday = data_frame[lstm_features].to_numpy()
    X_test_days = np.array(data_frame["Date"].to_list())
    y_test = np.array(data_frame["y-value"].to_list())
    
    # print(X_test[0])

    # print(data_frame.iloc[0]["LstmData"].loc[:,"Date"])
    # for i in range(len(data_frame)):
    #         print(f"Stock: {X_test_ticker[i]}, Date: {data_frame['Date'].iloc[i]}, y-value: {data_frame['y-value'].iloc[i]}")

    y_predict = best_model.predict([X_test, X_test_ticker, X_test_sector, X_test_yesterday])
    data_frame['y-predict'] = y_predict
    for ticker in data_frame["Orig_Ticker"].unique():
        data_frame.loc[data_frame["Orig_Ticker"] == ticker, 'y-predict-original'] = inverse_normalize_data(data_frame[['y-predict']], real_target, ticker)

    data_frame["Target-Val"] = target_val
    data_frame["Threshold"] = threshold
    data_frame["Target"] = tt
    data_frame = data_frame.assign(**threshold_levels.to_dict())
    data_frame["Score"] = score
    data_frame["Mode"] = mm
    data_frame["Predicted-Price"] = data_frame["Next-Day-Open"] * (1 + data_frame["Target-Val"])
    data_frame["Close-To-Open-Pct"] = data_frame["Next-Day-Close"] / data_frame["Next-Day-Open"]
    data_frame["High-To-Open-Pct"] = data_frame["Next-Day-High-To-Next-Day-Open-Ratio"]
    data_frame["Low-To-Open-Pct"] = data_frame["Next-Day-Low-To-Next-Day-Open-Ratio"]
    data_frame["y-value-balanced"] = data_frame["Next-Day-High-To-Next-Day-Open-Ratio"] >= data_frame["Target-Val"]
    
    data_frame['y-predict-binary'] = data_frame['y-predict-original'] >= threshold
    data_frame['y-predict-binary-with-5-percent-grace'] = data_frame['y-predict-original'] >= threshold * 0.95
    print(f"PRINTING POSITIVE PREDICTIONS FOR: {ss} WITH TARGET: {tt}")
    print(data_frame[data_frame['y-predict-binary'] == 1.0])
    
    to_write = data_frame[["Orig_Ticker", "Date", "y-value", "y-value-balanced", "y-predict", "y-predict-original", "y-predict-binary", "y-predict-binary-with-5-percent-grace" ,"Score", "Mode", "Threshold", "TPs", "FPs", "FNs", "Precision", "Recall", "Random-Guess", "Target", "Target-Val", "Next-Day-Open", "Predicted-Price", "Next-Day-Close", "Close-To-Open-Pct", "High-To-Open-Pct", "Low-To-Open-Pct"]]
    to_write.to_csv(save_location + "todays-guess.csv", mode='a', index=False, header=write_header)
    
    to_write_only_true = to_write[to_write["y-predict-binary"] == 1.0]
    to_write_only_true.to_csv(save_location + "todays-guess_only_true.csv", mode='a', index=False, header=write_header)
    
    to_write_only_true = to_write[to_write["y-predict-binary-with-5-percent-grace"] == 1.0]
    to_write_only_true.to_csv(save_location + "todays-guess_only_true_5_percent_grace.csv", mode='a', index=False, header=write_header)
    write_header = False

In [ ]:
print(data_frame.iloc[0]["LstmData"])

In [ ]:
data_frame = pd.read_csv(save_location + "todays-guess_only_true.csv")

filtered = data_frame[data_frame["Score"] >= 4]
filtered = filtered[filtered["y-predict-binary"] == True]
# Assign to the losses, that as if the market closed at the worst that the negative target value is hit.
filtered.loc[filtered['Close-To-Open-Pct'] < (1 - filtered['Target-Val']), 'Close-To-Open-Pct'] = (1 - filtered['Target-Val'])
filtered.loc[filtered['Close-To-Open-Pct'] < 0.985, 'Close-To-Open-Pct'] = 0.985
# filtered.loc[filtered['Low-To-Open-Pct'] < -0.0454, 'y-value-balanced'] = False
# # filtered.loc[filtered['Low-To-Open-Pct'] < -0.0454, 'Close-To-Open-Pct'] = 1-0.0454

gains = filtered[filtered["y-value-balanced"] == 1.0]["Target-Val"]
num_gains = gains.count()
total_gains = gains.sum()
num_loss = (filtered[filtered["y-value-balanced"] == 0.0]["Close-To-Open-Pct"] - 1.0).count()


losses = filtered[filtered["y-value-balanced"] == 0.0]

total_loss_uf = losses[losses["y-value-balanced"] == 0.0]["Close-To-Open-Pct"] - 1.0
total_loss = total_loss_uf.sum()

gains = gains+1
total_loss_uf = total_loss_uf + 1
# print(gains)

gains = (271.6504 * (gains**2)) - (519.3991 * (gains)) + 248.6636

# print(gains)
# print(f"\n\n\n\n\n")

# print(total_loss_uf)

losses = (271.6504 * (total_loss_uf**2)) -(519.3991 * (total_loss_uf)) + 248.6636

# print(losses)
# print(f"\n\n\n\n\n")

sum_ggg = gains.sum()
sum_lll = losses.sum()

#271.6504x2 −519.3991x+248.6636

print(f"{num_gains} Gains: {total_gains}, {num_loss} Loss: {total_loss}")
print(f"AVG Gain: {total_gains/ num_gains}, AVG Loss: {total_loss / num_loss}")
print(f"Total-GAIN: {sum_ggg}, Total-LOSS: {sum_lll}")
print(f"Started With: {num_loss + num_gains}, Ended with: {sum_ggg + sum_lll}, Calculated GAIN: {100 * ((sum_ggg + sum_lll) / (num_loss + num_gains)-1)}%")

In [ ]:
money = 100
filtered = filtered.sort_values(by='Date', ascending=True)
filtered = filtered.reset_index(drop=True)

for day in filtered["Date"].unique():
    trades = filtered[filtered["Date"] == day]
    per_invest = money / max(4,len(trades))
    invest_result = money - (per_invest * len(trades))
    trades = trades.reset_index(drop=True) 

    # print(trades)
    trades = trades.loc[trades.groupby(['Orig_Ticker', 'Target'])['Score'].idxmax()]
    # print(trades)
    trades = trades.sample(frac=1).drop_duplicates(subset=['Orig_Ticker', 'Target'])
    # print(trades)

    
    trades = trades.loc[trades.groupby(['Orig_Ticker'])['Score'].idxmax()]
    trades = trades.loc[trades.groupby(['Orig_Ticker'])['Target-Val'].idxmax()]
    # print(trades)
    trades = trades.sample(frac=1).drop_duplicates(subset=['Orig_Ticker'])
    # print(trades)

    trades = trades.reset_index(drop=True) 

    for index, trade in trades.iterrows():
        if trade["y-value-balanced"] == True:
            target = trade["Target-Val"] + 1
            # print(f"Target: {target}")
            gain = (271.6504 * (target**2)) - (519.3991 * (target)) + 248.6636
            # print(f"gain: {gain}")
            invest_result += per_invest * gain
        else:
            target = trade["Close-To-Open-Pct"]
            print(f"Target: {target}")
            loss = (271.6504 * (target**2)) - (519.3991 * (target)) + 248.6636
            print(f"loss: {loss}")
            invest_result += per_invest * loss
    print(f"END OF DAY:{day} TRADES: {len(trades)} MONEY: {invest_result} STOCKS: {trades['Orig_Ticker'].unique()}")
    money = invest_result
        
print(money)

In [ ]:
x = [11,2,3,4,5,6,7,8,9,10]

df = pd.DataFrame({'x': x})

val = df['x'].rolling(window=5, min_periods=1).max().shift(-5)
print(val)

In [ ]:
# data_frame["y-predict"] = y_predict

# sector_threshold_mapping =  pd.read_csv(model_location + model_name + ".csv")
# print(sector_threshold_mapping)
# for sector in data_frame["Orig_Sector"].unique():
#     threshold = sector_threshold_mapping[sector_threshold_mapping["Sector"] == sector].iloc[0]["Threshold"]
#     data_frame.loc[data_frame["Orig_Sector"] == sector, "y-predict-binary"] = (data_frame[data_frame["Orig_Sector"] == sector]["y-predict"] > threshold).astype(float)

# print(f"\n\nResults: \n {data_frame.loc[(data_frame['y-predict-binary'] == 1) & ((data_frame['Target-20D-Mean'] + data_frame['Target-20D-Sigma']) > 0.011)]}")


In [ ]:
import numpy as np
X = np.load('X_values.npy')
y = np.load('y_values.npy')
dates = np.load('dates_values.npy', allow_pickle=True)
tickers = np.load('tickers_values.npy', allow_pickle=True)

In [ ]:
import pandas as pd


index = 1100
print(sum(y))

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(X.shape)
    print(dates.shape)
    print(tickers.shape)
    arr = X[index,0:64,3]
    print(dates[index])
    print(tickers[index])
    print(y[index])
    for i in arr:
        print(i)
    # print(arr.min())